In [1]:
import pandas as pd
import numpy as np

In [5]:
patient_df = pd.read_csv("Patient_Data.csv")
billing_df = pd.read_csv("Billing_Data.csv")

print(patient_df.head())


   PatientID     Name   Department     Doctor  BillAmount  ReceptionistID  \
0        101    Alice   Cardiology  Dr. Smith      5000.0               1   
1        102      Bob    Neurology   Dr. John         NaN               2   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0               1   
3        104    David   Cardiology  Dr. Smith      6200.0               3   
4        105      Eva  Dermatology   Dr. Rose         NaN               2   

        CheckInTime  
0  2023-01-10 09:00  
1  2023-01-11 10:30  
2  2023-01-12 11:00  
3  2023-01-13 12:00  
4  2023-01-14 08:45  


In [6]:
print(patient_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes
None


In [7]:
patient_df.describe(include='all')

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
count,6.000000,6,6,6,4.000000,6.000000,6
unique,NaN,5,4,4,NaN,NaN,5
top,NaN,Alice,Cardiology,Dr. Smith,NaN,NaN,2023-01-10 09:00
freq,NaN,2,3,3,NaN,NaN,2
mean,102.666667,NaN,NaN,NaN,5925.000000,1.666667,NaN
std,1.632993,NaN,NaN,NaN,1192.686044,0.816497,NaN
min,101.000000,NaN,NaN,NaN,5000.000000,1.000000,NaN
25%,101.250000,NaN,NaN,NaN,5000.000000,1.000000,NaN
50%,102.500000,NaN,NaN,NaN,5600.000000,1.500000,NaN
75%,103.750000,NaN,NaN,NaN,6525.000000,2.000000,NaN


In [9]:
billing_related = patient_df[
    ['PatientID', 'Department', 'Doctor', 'BillAmount']
]
billing_related.head()

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN


In [11]:
# Drop unnecessary columns
patient_cleaned = patient_df.drop(
    columns=['ReceptionistID', 'CheckInTime'],
    errors='ignore'     # prevents error if columns don't exist
)
print(patient_cleaned.head())

   PatientID     Name   Department     Doctor  BillAmount
0        101    Alice   Cardiology  Dr. Smith      5000.0
1        102      Bob    Neurology   Dr. John         NaN
2        103  Charlie  Orthopedics    Dr. Lee      7500.0
3        104    David   Cardiology  Dr. Smith      6200.0
4        105      Eva  Dermatology   Dr. Rose         NaN


In [12]:
department_bill = patient_cleaned.groupby(
    'Department'
)['BillAmount'].sum()

print(department_bill)

Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


In [13]:
patient_no_duplicates = patient_cleaned.drop_duplicates(
    subset='PatientID'
)

print(patient_no_duplicates.head())
print(patient_no_duplicates.shape)


   PatientID     Name   Department     Doctor  BillAmount
0        101    Alice   Cardiology  Dr. Smith      5000.0
1        102      Bob    Neurology   Dr. John         NaN
2        103  Charlie  Orthopedics    Dr. Lee      7500.0
3        104    David   Cardiology  Dr. Smith      6200.0
4        105      Eva  Dermatology   Dr. Rose         NaN
(5, 5)


In [14]:
mean_bill = patient_no_duplicates['BillAmount'].mean()

# Fill missing values
patient_no_duplicates['BillAmount'] = (
    patient_no_duplicates['BillAmount']
    .fillna(mean_bill)
)

print("\nMISSING VALUES FILLED WITH MEAN")
print(patient_no_duplicates['BillAmount'].isnull().sum())


MISSING VALUES FILLED WITH MEAN
0


C:\Users\Hp\AppData\Local\Temp\ipykernel_12820\2828278124.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  patient_no_duplicates['BillAmount'] = (


In [15]:
merged_df = pd.merge(
    patient_no_duplicates,
    billing_df,
    on='PatientID',
    how='inner'
)

print("\nMERGED DATASET")
print(merged_df.head())


MERGED DATASET
   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000              2000   
1        102      Bob    Neurology   Dr. John  6233.333333              1500   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000              2500   
3        104    David   Cardiology  Dr. Smith  6200.000000              3000   
4        105      Eva  Dermatology   Dr. Rose  6233.333333              1000   

   FinalAmount  
0         3000  
1         3500  
2         5000  
3         3200  
4         4000  


In [16]:
new_patients = pd.DataFrame({
    'PatientID': [1001, 1002],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor': ['Dr. Sharma', 'Dr. Rao'],
    'BillAmount': [4500, 5200]
})

# Row-wise concatenation
updated_patient_df = pd.concat(
    [patient_no_duplicates, new_patients],
    ignore_index=True
)

print("\nDATA AFTER ADDING NEW PATIENTS")
print(updated_patient_df.tail())



DATA AFTER ADDING NEW PATIENTS
   PatientID     Name   Department      Doctor   BillAmount
2        103  Charlie  Orthopedics     Dr. Lee  7500.000000
3        104    David   Cardiology   Dr. Smith  6200.000000
4        105      Eva  Dermatology    Dr. Rose  6233.333333
5       1001      NaN   Cardiology  Dr. Sharma  4500.000000
6       1002      NaN    Neurology     Dr. Rao  5200.000000


In [17]:
additional_columns = pd.DataFrame({
    'InsuranceCovered': [True] * len(updated_patient_df),
    'FinalAmount': updated_patient_df['BillAmount'] * 0.9
})

# Column-wise concatenation
final_dataset = pd.concat(
    [updated_patient_df, additional_columns],
    axis=1
)

print("\nFINAL DATASET")
print(final_dataset.head())



FINAL DATASET
   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000              True   
1        102      Bob    Neurology   Dr. John  6233.333333              True   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000              True   
3        104    David   Cardiology  Dr. Smith  6200.000000              True   
4        105      Eva  Dermatology   Dr. Rose  6233.333333              True   

   FinalAmount  
0       4500.0  
1       5610.0  
2       6750.0  
3       5580.0  
4       5610.0  


In [22]:
print("\nFINAL DATASET INFO:")
print(final_dataset.info())

print("\nTOTAL MISSING VALUES:")
print(final_dataset.isnull().sum())

print("\nFINAL DATASET SHAPE:")
print(final_dataset.shape)


FINAL DATASET INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PatientID         7 non-null      int64  
 1   Name              5 non-null      object 
 2   Department        7 non-null      object 
 3   Doctor            7 non-null      object 
 4   BillAmount        7 non-null      float64
 5   InsuranceCovered  7 non-null      bool   
 6   FinalAmount       7 non-null      float64
dtypes: bool(1), float64(2), int64(1), object(3)
memory usage: 475.0+ bytes
None

TOTAL MISSING VALUES:
PatientID           0
Name                2
Department          0
Doctor              0
BillAmount          0
InsuranceCovered    0
FinalAmount         0
dtype: int64

FINAL DATASET SHAPE:
(7, 7)
